# Stage 6 — Intervention Design

This notebook translates the final XGBoost model's predicted probabilities into transparent, data-supported risk tiers and care-transition recommendations. It is a decision-support prototype; clinical governance is required before deployment.

## Load model and score the test cohort

In [1]:
from pathlib import Path

import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
RESULTS_DIR = PROJECT_ROOT / 'results'
DEPLOYMENT_THRESHOLD = 0.12

pipeline = joblib.load(RESULTS_DIR / 'xgboost.joblib')
X_test = pd.read_parquet(DATA_DIR / 'X_test.parquet')
y_test = pd.read_parquet(DATA_DIR / 'y_test.parquet')['readmitted_30']
probabilities = pipeline.predict_proba(X_test)[:, 1]
print(f'Scored {len(X_test):,} test patients; probability range: {probabilities.min():.4f}–{probabilities.max():.4f}')

Scored 14,997 test patients; probability range: 0.0146–0.6422


## Data-driven risk-tier boundaries

The candidate bins begin at the deployed high-risk flag of 0.12. Observed, rather than predicted, readmission rates are inspected to identify meaningful rises in realised risk.

In [2]:
candidate_bins = [0.00, 0.12, 0.20, 0.35, 0.50, 1.000001]
candidate_labels = ['[0.00, 0.12)', '[0.12, 0.20)', '[0.20, 0.35)', '[0.35, 0.50)', '[0.50, 1.00]']
candidate_summary = (
    pd.DataFrame({'probability_bin': pd.cut(probabilities, candidate_bins, right=False, labels=candidate_labels), 'readmitted_30': y_test})
    .groupby('probability_bin', observed=False)['readmitted_30']
    .agg(support='size', observed_readmission_rate='mean')
    .reset_index()
)
candidate_summary['rate_change_from_prior'] = candidate_summary['observed_readmission_rate'].diff()
print('Candidate probability-bin evidence:')
print(candidate_summary.to_string(index=False, float_format=lambda value: f'{value:.4f}'))

# The observed rate rises by roughly 9 percentage points at 0.20 and 19 points at 0.35.
# Above 0.35, the 0.50+ bin is tiny (n=31), so it is combined with 0.35–0.50.
TIER_BOUNDARIES = (0.12, 0.20, 0.35)
TIER_ORDER = ['Low Risk', 'Moderate Risk', 'High Risk', 'Very High Risk']
tier_ranges = {
    'Low Risk': '[0.00, 0.12)',
    'Moderate Risk': '[0.12, 0.20)',
    'High Risk': '[0.20, 0.35)',
    'Very High Risk': '[0.35, 1.00]',
}

def assign_risk_tier(probability: float) -> str:
    """Return the intervention tier for one predicted readmission probability."""
    if probability < TIER_BOUNDARIES[0]:
        return 'Low Risk'
    if probability < TIER_BOUNDARIES[1]:
        return 'Moderate Risk'
    if probability < TIER_BOUNDARIES[2]:
        return 'High Risk'
    return 'Very High Risk'

tier_assignments = pd.Series([assign_risk_tier(float(probability)) for probability in probabilities], name='tier')
tier_summary = (
    pd.DataFrame({'tier': tier_assignments, 'readmitted_30': y_test})
    .groupby('tier', observed=False)['readmitted_30']
    .agg(support='size', observed_readmission_rate='mean')
    .reindex(TIER_ORDER)
    .reset_index()
)
tier_summary['probability_range'] = tier_summary['tier'].map(tier_ranges)
tier_summary = tier_summary[['tier', 'probability_range', 'support', 'observed_readmission_rate']]
print('\nFinal tier evidence:')
print(tier_summary.to_string(index=False, float_format=lambda value: f'{value:.4f}'))

Candidate probability-bin evidence:
probability_bin  support  observed_readmission_rate  rate_change_from_prior
   [0.00, 0.12)     8786                     0.0687                     NaN
   [0.12, 0.20)     4367                     0.1397                  0.0709
   [0.20, 0.35)     1584                     0.2285                  0.0889
   [0.35, 0.50)      229                     0.4192                  0.1907
   [0.50, 1.00]       31                     0.3871                 -0.0321

Final tier evidence:
          tier probability_range  support  observed_readmission_rate
      Low Risk      [0.00, 0.12)     8786                     0.0687
 Moderate Risk      [0.12, 0.20)     4367                     0.1397
     High Risk      [0.20, 0.35)     1584                     0.2285
Very High Risk      [0.35, 1.00]      260                     0.4154


## Tier-specific care-transition recommendations

These recommendations are literature-grounded workflow examples. Project BOOST emphasizes structured discharge preparation, teach-back, follow-up, and care coordination; the Care Transitions Intervention (CTI) uses a transition coach, medication management, follow-up, and warning-sign education over the post-discharge period.

In [3]:
interventions = {
    'Low Risk': ('Standard discharge instructions and routine primary-care follow-up; consistent with Project BOOST risk stratification, no additional intensive transition service is assigned when observed risk is low.'),
    'Moderate Risk': ('Place a post-discharge follow-up call within 72 hours and complete a medication-reconciliation review; this is a focused Project BOOST-style transition support step.'),
    'High Risk': ('Use Project BOOST-style structured discharge planning: teach-back education, a follow-up appointment scheduled before discharge, and a phone call within 48 hours.'),
    'Very High Risk': ('Use the full Care Transitions Intervention (CTI): assign a transition coach, arrange a home visit within 72 hours, reconcile medications, teach red-flag symptoms, and conduct 30-day check-in calls.'),
}

def get_intervention(tier: str) -> str:
    """Return the recommended care-transition intervention for a named tier."""
    if tier not in interventions:
        raise ValueError(f'Unknown tier: {tier}')
    return interventions[tier]

for tier in TIER_ORDER:
    print(f'{tier}: {get_intervention(tier)}')

print('\nFunction checks:')
for example_probability in [0.05, 0.12, 0.24, 0.42]:
    tier = assign_risk_tier(example_probability)
    print(f'{example_probability:.2f} -> {tier} -> {get_intervention(tier)}')

Low Risk: Standard discharge instructions and routine primary-care follow-up; consistent with Project BOOST risk stratification, no additional intensive transition service is assigned when observed risk is low.
Moderate Risk: Place a post-discharge follow-up call within 72 hours and complete a medication-reconciliation review; this is a focused Project BOOST-style transition support step.
High Risk: Use Project BOOST-style structured discharge planning: teach-back education, a follow-up appointment scheduled before discharge, and a phone call within 48 hours.
Very High Risk: Use the full Care Transitions Intervention (CTI): assign a transition coach, arrange a home visit within 72 hours, reconcile medications, teach red-flag symptoms, and conduct 30-day check-in calls.

Function checks:
0.05 -> Low Risk -> Standard discharge instructions and routine primary-care follow-up; consistent with Project BOOST risk stratification, no additional intensive transition service is assigned when obs

## Intervention flow diagram

In [4]:
fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 14)
ax.set_ylim(0, 7)
ax.axis('off')

def add_box(x, y, width, height, text, color):
    box = FancyBboxPatch((x, y), width, height, boxstyle='round,pad=0.03', facecolor=color, edgecolor='#2C3E50', linewidth=1.5)
    ax.add_patch(box)
    ax.text(x + width / 2, y + height / 2, text, ha='center', va='center', wrap=True, fontsize=9)

add_box(0.4, 3.0, 2.2, 1.0, 'Patient data\n(discharge record)', '#D6EAF8')
add_box(3.5, 3.0, 2.2, 1.0, 'XGBoost prediction\n(probability)', '#D5F5E3')
add_box(6.6, 3.0, 2.2, 1.0, 'Data-driven\ntier assignment', '#FCF3CF')
for x_start in [2.6, 5.7]:
    ax.annotate('', xy=(x_start + 0.8, 3.5), xytext=(x_start, 3.5), arrowprops=dict(arrowstyle='->', lw=1.8))

tier_y = [5.35, 3.95, 2.55, 1.15]
tier_colors = ['#EAF2F8', '#D4E6F1', '#FADBD8', '#F5B7B1']
diagram_interventions = {
    'Low Risk': 'Standard instructions\nroutine PCP follow-up',
    'Moderate Risk': 'Call within 72 hours\nmedication reconciliation',
    'High Risk': 'Project BOOST: teach-back\nappointment before discharge\ncall within 48 hours',
    'Very High Risk': 'CTI: coach + home visit ≤72h\nmedications • red flags\n30-day check-in calls',
}
for tier, y, color in zip(TIER_ORDER, tier_y, tier_colors):
    add_box(10.0, y, 3.5, 1.05, f"{tier} {tier_ranges[tier]}\n{diagram_interventions[tier]}", color)
    ax.annotate('', xy=(10.0, y + 0.525), xytext=(8.8, 3.5), arrowprops=dict(arrowstyle='->', lw=1.2))

ax.set_title('Readmission-Risk Intervention Flow', fontsize=15, pad=15)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'intervention_tier_diagram.png', dpi=300, bbox_inches='tight')
plt.show()

C:\Users\sagir\AppData\Local\Temp\ipykernel_20744\1567120324.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Tier distribution and realised risk

In [5]:
fig, left_axis = plt.subplots(figsize=(10, 6))
positions = np.arange(len(tier_summary))
support_bars = left_axis.bar(positions - 0.2, tier_summary['support'], width=0.4, color='#5DADE2', label='Patients')
left_axis.set_ylabel('Number of patients')
left_axis.set_xticks(positions)
left_axis.set_xticklabels(tier_summary['tier'])

right_axis = left_axis.twinx()
risk_bars = right_axis.bar(positions + 0.2, tier_summary['observed_readmission_rate'], width=0.4, color='#E74C3C', label='Observed readmission rate')
right_axis.set_ylabel('Observed readmission rate')
right_axis.set_ylim(0, max(tier_summary['observed_readmission_rate']) * 1.3)
right_axis.yaxis.set_major_formatter('{x:.0%}')
for bar, value in zip(risk_bars, tier_summary['observed_readmission_rate']):
    right_axis.text(bar.get_x() + bar.get_width() / 2, value, f'{value:.1%}', ha='center', va='bottom')
left_axis.set_title('Risk-Tier Distribution and Observed 30-Day Readmission')
left_axis.legend(loc='upper left')
right_axis.legend(loc='upper right')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'tier_distribution_and_risk.png', dpi=300, bbox_inches='tight')
plt.show()

C:\Users\sagir\AppData\Local\Temp\ipykernel_20744\3254663309.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Export tier definitions

References: Society of Hospital Medicine, *Project BOOST Implementation Guide*; Care Transitions Intervention, *The Model*. Clinical teams should adapt these workflows to local resources and governance requirements.

In [6]:
tier_definitions = tier_summary.copy()
tier_definitions['intervention'] = tier_definitions['tier'].map(interventions)
tier_definitions.to_csv(RESULTS_DIR / 'intervention_tier_definitions.csv', index=False)
print(tier_definitions.to_string(index=False, float_format=lambda value: f'{value:.4f}'))
print(f"\nSaved: {RESULTS_DIR / 'intervention_tier_definitions.csv'}")

          tier probability_range  support  observed_readmission_rate                                                                                                                                                                                             intervention
      Low Risk      [0.00, 0.12)     8786                     0.0687 Standard discharge instructions and routine primary-care follow-up; consistent with Project BOOST risk stratification, no additional intensive transition service is assigned when observed risk is low.
 Moderate Risk      [0.12, 0.20)     4367                     0.1397                                    Place a post-discharge follow-up call within 72 hours and complete a medication-reconciliation review; this is a focused Project BOOST-style transition support step.
     High Risk      [0.20, 0.35)     1584                     0.2285                                       Use Project BOOST-style structured discharge planning: teach-back education, a foll

## Tier-boundary confusion matrices

The four tiers are separated by three escalation boundaries. For each lower threshold below, a patient is treated as positive when the predicted probability is at or above that threshold; the true positive class is a 30-day readmission. Low Risk is the baseline tier, so it has no lower escalation boundary of its own.

In [7]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score

tier_boundary_specs = [
    (0.12, 'Moderate Risk or higher escalation'),
    (0.20, 'High Risk or higher escalation'),
    (0.35, 'Very High Risk escalation'),
]

confusion_records = []
for threshold, escalation_label in tier_boundary_specs:
    tier_flag = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, tier_flag, labels=[0, 1]).ravel()
    precision = precision_score(y_test, tier_flag, zero_division=0)
    recall = recall_score(y_test, tier_flag, zero_division=0)
    confusion_records.append({
        'threshold': threshold,
        'escalation_rule': escalation_label,
        'true_negatives': tn,
        'false_positives': fp,
        'false_negatives': fn,
        'true_positives': tp,
        'precision': precision,
        'recall': recall,
    })
    print(
        f'At the {threshold:.2f} boundary ({escalation_label}), the escalation flag has '
        f'precision {precision:.4f} and recall {recall:.4f} '
        f'(TP={tp:,}, FP={fp:,}, TN={tn:,}, FN={fn:,}).'
    )

tier_confusion_matrices = pd.DataFrame(confusion_records)
display(tier_confusion_matrices)

output_path = RESULTS_DIR / 'tier_confusion_matrices.csv'
tier_confusion_matrices.to_csv(output_path, index=False)
print(f'\nSaved: {output_path}')

At the 0.12 boundary (Moderate Risk or higher escalation), the escalation flag has precision 0.1739 and recall 0.6413 (TP=1,080, FP=5,131, TN=8,182, FN=604).
At the 0.20 boundary (High Risk or higher escalation), the escalation flag has precision 0.2549 and recall 0.2791 (TP=470, FP=1,374, TN=11,939, FN=1,214).
At the 0.35 boundary (Very High Risk escalation), the escalation flag has precision 0.4154 and recall 0.0641 (TP=108, FP=152, TN=13,161, FN=1,576).


,threshold,escalation_rule,true_negatives,false_positives,false_negatives,true_positives,precision,recall
0,0.12,Moderate Risk or higher escalation,8182,5131,604,1080,0.173885,0.641330
1,0.20,High Risk or higher escalation,11939,1374,1214,470,0.254881,0.279097
2,0.35,Very High Risk escalation,13161,152,1576,108,0.415385,0.064133



Saved: C:\Users\sagir\OneDrive\Desktop\capstone\results\tier_confusion_matrices.csv
